# 3. Construcción de Agentes con LangChain

## Objetivos de Aprendizaje
- Comprender las ventajas de usar un framework como LangChain para el desarrollo de agentes.
- Aprender a definir herramientas (Tools) de forma sencilla con el decorador `@tool`.
- Crear un agente utilizando el constructor `create_openai_tools_agent`.
- Ejecutar el agente y gestionar la interacción mediante el `AgentExecutor`.

## ¿Por qué usar LangChain?

En los notebooks anteriores, construimos agentes desde cero. Primero, parseando texto, y luego, usando `function calling` nativo. Aunque el `function calling` es una gran mejora, todavía teníamos que:

1.  **Gestionar el historial de mensajes (`messages`) manualmente**: Añadir cada respuesta del usuario, del asistente y de la herramienta a la lista.
2.  **Orquestar el flujo de llamadas**: Escribir la lógica `if/else` para decidir si llamar a una herramienta, ejecutarla y volver a llamar al modelo.
3.  **Formatear las herramientas**: Escribir el JSON Schema para cada herramienta, lo cual es propenso a errores.

**LangChain** es un framework que abstrae toda esta complejidad. Actúa como una capa intermedia que simplifica enormemente la creación de aplicaciones basadas en LLMs, incluyendo los agentes. 

**Ventajas clave:**
- **Componentes Modulares**: Ofrece piezas reutilizables (LLMs, Prompts, Herramientas, etc.) que se pueden ensamblar fácilmente.
- **Agentes Listos para Usar**: Proporciona abstracciones de alto nivel como el `AgentExecutor` que manejan el ciclo de razonamiento (ReAct) por nosotros.
- **Integraciones**: Se conecta con cientos de fuentes de datos, APIs y otros servicios de forma nativa.

### 1. Instalación y Configuración

In [1]:
# Instalación de dependencias.
#
# Solo hace falta en Google Colab. En local, `uv sync` ya instaló todo esto con las
# versiones exactas del uv.lock; lanzar pip con -U aquí las actualizaría y rompería
# la reproducibilidad que el curso garantiza a todo el grupo.
import sys

if "google.colab" in sys.modules:
    %pip install -qU langchain-groq groq langgraph langchain langchain-classic requests python-dotenv
else:
    print("Entorno local: las dependencias ya las instaló uv sync.")


Entorno local: las dependencias ya las instaló uv sync.


In [2]:
import os
import re
import requests
from urllib.parse import quote
from langchain_groq import ChatGroq

# Carga de credenciales: funciona igual en Google Colab y en local (.env)
try:
    from google.colab import userdata  # type: ignore
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

assert os.getenv("GROQ_API_KEY"), "Falta GROQ_API_KEY (Colab: Secrets · local: archivo .env)"

# Agente de LangChain con herramientas nativas (create_openai_tools_agent).
# Por esta vía el modelo grande es el fiable: medido sobre 6 consultas, llama-3.3-70b
# acertó 6/6 el formato de la llamada a la función y llama-3.1-8b solo 4/6.
# (Ojo: con el SDK crudo de Groq la relación se invierte; ver 2-agent-function-calling.)
MODELO = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")

WIKIPEDIA_API_URL = "https://es.wikipedia.org/w/api.php"
WIKIPEDIA_SUMMARY_URL = "https://es.wikipedia.org/api/rest_v1/page/summary"
WIKIPEDIA_HEADERS = {
    "User-Agent": "Curso-IA-DUOC/1.0 (notebook educativo; contacto: estudiante@example.com)"
}

# --- Configuración del LLM con LangChain ---
# ChatGroq es el "envoltorio" (wrapper) de LangChain sobre la API de Groq.
# Lee GROQ_API_KEY del entorno automáticamente: no hace falta pasar api_key.
try:
    llm = ChatGroq(
        model=MODELO,
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

✅ LLM de LangChain configurado.


### 2. Definición de Herramientas con LangChain

LangChain simplifica enormemente la creación de herramientas. En lugar de escribir un JSON Schema manual, simplemente usamos el decorador `@tool` sobre una función de Python. LangChain se encarga de inferir el esquema a partir de la firma de la función y su docstring.

El docstring es **muy importante**, ya que se usa como la descripción que el LLM ve para decidir si usar la herramienta o no.

In [3]:
from langchain_classic.agents import tool


def _wikipedia_get_json(url, params=None):
    """Solicita JSON a Wikipedia y devuelve un error legible si la respuesta no es válida."""
    try:
        response = requests.get(
            url,
            params=params,
            headers=WIKIPEDIA_HEADERS,
            timeout=10,
        )
        response.raise_for_status()
        return response.json(), None
    except requests.exceptions.JSONDecodeError:
        return None, "Wikipedia devolvió una respuesta vacía o no válida en JSON."
    except requests.exceptions.RequestException as e:
        return None, f"No se pudo consultar Wikipedia: {e}"


def _limitar_oraciones(texto, max_oraciones=2):
    oraciones = re.split(r"(?<=[.!?])\s+", texto.strip())
    return " ".join(oraciones[:max_oraciones])


@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Es ideal para obtener información sobre personas, lugares o conceptos históricos y científicos."""
    if not query or not query.strip():
        return "No se recibió un término de búsqueda para Wikipedia."

    search_data, error = _wikipedia_get_json(
        WIKIPEDIA_API_URL,
        params={
            "action": "query",
            "list": "search",
            "srsearch": query,
            "srlimit": 1,
            "format": "json",
            "utf8": 1,
        },
    )
    if error:
        return error

    results = search_data.get("query", {}).get("search", [])
    if not results:
        return f"No se encontró ninguna página para '{query}'."

    title = results[0]["title"]
    summary_data, error = _wikipedia_get_json(
        f"{WIKIPEDIA_SUMMARY_URL}/{quote(title)}"
    )
    if error:
        return error

    extract = summary_data.get("extract")
    if not extract:
        return f"Wikipedia no entregó un resumen disponible para '{title}'."

    return _limitar_oraciones(extract, max_oraciones=2)

tools = [get_wikipedia_summary]

print("✅ Herramientas de LangChain definidas.")

✅ Herramientas de LangChain definidas.


### 3. Creación del Agente

Para crear el agente, necesitamos dos cosas:

1.  **Un Prompt**: Una plantilla que le dice al agente cómo razonar y cómo usar las herramientas. LangChain ya tiene prompts pre-construidos y optimizados para esto. Usaremos `hub.pull("hwchase17/openai-functions-agent")` para obtener una plantilla probada.
2.  **El Agente en sí**: Usamos la función `create_openai_tools_agent`, que une el LLM, las herramientas y el prompt.

El resultado es un `Runnable` de LangChain, que es el agente listo para ser ejecutado.

In [4]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.agents import create_openai_tools_agent

# Descargar un prompt pre-diseñado y optimizado para agentes con function calling
# El prompt del agente, definido aquí en vez de descargarlo del hub de LangChain.
#
# Antes esto era `hub.pull("hwchase17/openai-tools-agent")`. Se cambió por tres razones:
#   1. Seguridad: descargar un prompt público deserializa objetos de LangChain de un
#      tercero. LangChain lo bloqueó por defecto justo por eso.
#   2. Fiabilidad: `hub.pull` necesita red; sin internet el notebook no arranca.
#   3. Didáctica: así ves el prompt real que gobierna al agente, que es lo interesante.
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

# Unir el LLM, las herramientas y el prompt para crear el agente
agent = create_openai_tools_agent(llm, tools, prompt)

print("✅ Agente de LangChain creado.")

✅ Agente de LangChain creado.


### 4. Ejecución del Agente con `AgentExecutor`

El `agent` que creamos en el paso anterior es solo el "cerebro". Sabe cómo razonar, pero no puede ejecutar el ciclo de `Pensamiento -> Acción -> Observación` por sí mismo.

Para eso, usamos el **`AgentExecutor`**. Esta clase toma el agente y las herramientas, y se encarga de toda la orquestación:

- Llama al agente con la entrada del usuario.
- Si el agente decide usar una herramienta, el `AgentExecutor` la ejecuta.
- Pasa el resultado de la herramienta de vuelta al agente.
- Repite el proceso hasta que el agente da una respuesta final.
- Gestiona el historial de la conversación (`chat_history`).

In [5]:
from langchain_classic.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ AgentExecutor listo para funcionar.")

✅ AgentExecutor listo para funcionar.


### 5. Invocando al Agente

Ahora, simplemente llamamos al método `invoke` del `agent_executor` con la pregunta. El parámetro `chat_history` es opcional pero útil para conversaciones de seguimiento.

In [6]:
query = "¿Quién fue Marie Curie y cuáles fueron sus logros más importantes?"

response = agent_executor.invoke({
    "input": query,
    "chat_history": []
})

print(f"🏁 Respuesta final del agente: {response['output']}")



> Entering new AgentExecutor chain...



Invoking: `get_wikipedia_summary` with `{'query': 'Marie Curie'}`




Maria Salomea Skłodowska-Curie, más conocida como Marie Curie o Madame Curie, fue una física y química polaca, luego naturalizada francesa. Pionera en el campo de la radiactividad, es la primera y única persona en recibir dos premios Nobel en distintas especialidades científicas: Física y Química.


Invoking: `get_wikipedia_summary` with `{'query': 'logros de Marie Curie'}`




Maria Salomea Skłodowska-Curie, más conocida como Marie Curie o Madame Curie, fue una física y química polaca, luego naturalizada francesa. Pionera en el campo de la radiactividad, es la primera y única persona en recibir dos premios Nobel en distintas especialidades científicas: Física y Química.

Marie Curie fue una científica polaca-francesa que hizo importantes contribuciones a la física y la química. Sus logros más destacados incluyen:

1. Descubrimiento de los elementos radiactivos polonio y radio: Marie Curie, junto con su esposo Pierre Curie, descubrió los elementos radiactivos polonio y radio en 1898. Este descubrimiento revolucionó la comprensión de la radiactividad y abrió nuevas áreas de investigación en la física y la química.
2. Desarrollo de la teoría de la radiactividad: Marie Curie desarrolló la teoría de la radiactividad, que explica cómo los átomos emiten radiación. Su trabajo en este campo sentó las bases para la comprensión de la estructura atómica y la naturaleza de la radiación.
3. Premios Nobel: Marie Curie fue la primera persona en recibir dos premios Nobel, uno en Física en 1903 y otro en Química en 1911. Su premio Nobel de Física fue por su trabajo en la radiactividad, y su premio Nobel de Química fue por su trabajo en la isolación de los elementos radi

## Conclusiones

Como hemos visto, LangChain reduce drásticamente la cantidad de código repetitivo y la complejidad de construir un agente. Nos hemos podido centrar en:

1.  **Definir la lógica de la herramienta**: La función `get_wikipedia_summary`.
2.  **Ensamblar los componentes**: Unir el LLM, las herramientas y un prompt usando las abstracciones de LangChain.

El `AgentExecutor` se encargó de todo el ciclo de ejecución, el manejo de estado y la orquestación, que antes teníamos que programar manualmente.

En el siguiente notebook, exploraremos **CrewAI**, otro framework de alto nivel que se especializa en la creación de equipos de agentes que colaboran para resolver tareas aún más complejas.